### 프로젝트 목표

한국어 Q&A 데이터를 활용하여 데이터 전처리부터 토크나이징, Transformer 아키텍처 설계 및 학습까지 전체 NLP 파이프라인을 밑바닥부터(From Scratch) 구축

In [1]:
!pip install gensim

In [2]:
!pip install nltk

In [3]:
# ==========================================================
# 1. 라이브러리 import
# ==========================================================

# ----------------------------------------------------------
# 파일 및 폴더 경로 관리
# ----------------------------------------------------------
import os


# ----------------------------------------------------------
# 데이터 처리 라이브러리
# ----------------------------------------------------------
# DataFrame 형태로 CSV 데이터를 처리하기 위해 사용
import pandas as pd

# 행렬 연산 및 수치 계산을 위해 사용
import numpy as np


# ----------------------------------------------------------
# PyTorch 딥러닝 라이브러리
# ----------------------------------------------------------
# Tensor 연산 및 GPU 사용
import torch

# 신경망 Layer 구현
import torch.nn as nn

# Optimizer 구현
import torch.optim as optim


# ----------------------------------------------------------
# Dataset / DataLoader
# ----------------------------------------------------------
# 학습 데이터를 배치 단위로 전달하기 위해 사용
from torch.utils.data import Dataset, DataLoader


# ----------------------------------------------------------
# SentencePiece Tokenizer
# ----------------------------------------------------------
# 자연어 문장을 Token ID 형태로 변환하기 위해 사용
import sentencepiece as spm


# ----------------------------------------------------------
# Word2Vec
# ----------------------------------------------------------
# 데이터 증강(Lexical Substitution)을 위해 사용
from gensim.models import Word2Vec


# ----------------------------------------------------------
# 진행률 표시
# ----------------------------------------------------------
from tqdm import tqdm


# ----------------------------------------------------------
# 문자열 전처리
# ----------------------------------------------------------
import re


# ----------------------------------------------------------
# 랜덤 데이터 처리
# ----------------------------------------------------------
import random


# ----------------------------------------------------------
# BLEU Score 평가
# ----------------------------------------------------------
# 생성된 답변과 실제 답변의 유사도를 평가하기 위해 사용
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

In [4]:
# ==========================================================
# 2. Random Seed 설정
# ==========================================================

# 동일한 결과 재현을 위해 난수 고정

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)


# GPU 사용 시 Seed 고정
if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)


print("Random Seed 설정 완료")

Random Seed 설정 완료


In [5]:
# ==========================================================
# 3. 학습 장치 설정
# ==========================================================

# GPU 사용 가능하면 CUDA 사용
# 불가능하면 CPU 사용

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)


print("현재 사용 장치 :", device)

현재 사용 장치 : cuda


In [6]:
# ==========================================================
# 4. 데이터 경로 설정
# ==========================================================


# Chatbot 데이터 CSV 경로

DATA_PATH = os.path.join(
    "data",
    "ChatbotData.csv"
)



# Word2Vec 사전학습 모델 경로

W2V_PATH = os.path.join(
    "data",
    "ko.bin"
)



print("데이터 경로 :", DATA_PATH)

print("Word2Vec 경로 :", W2V_PATH)

데이터 경로 : data/ChatbotData.csv
Word2Vec 경로 : data/ko.bin


In [7]:
# ==========================================================
# 5. 파일 존재 확인
# ==========================================================


if os.path.exists(DATA_PATH):

    print("ChatbotData.csv 파일 확인 완료")

else:

    print("ChatbotData.csv 파일을 찾을 수 없습니다.")



if os.path.exists(W2V_PATH):

    print("ko.bin 파일 확인 완료")

else:

    print("ko.bin 파일을 찾을 수 없습니다.")

ChatbotData.csv 파일 확인 완료
ko.bin 파일 확인 완료


In [8]:
# ==========================================================
# 6. 데이터 로드
# ==========================================================


# CSV 파일 읽기

data = pd.read_csv(DATA_PATH)



# 데이터 개수 확인

print("전체 데이터 개수 :", len(data))



# 데이터 상위 5개 확인

data.head()

전체 데이터 개수 : 11823


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [9]:
# ==========================================================
# 7. 데이터 정보 확인
# ==========================================================


# 컬럼 확인

print(data.columns)



# 결측치 확인

print("\n결측치 확인")

print(data.isnull().sum())



# 데이터 타입 확인

print("\n데이터 타입")

print(data.dtypes)

Index(['Q', 'A', 'label'], dtype='object')

결측치 확인
Q        0
A        0
label    0
dtype: int64

데이터 타입
Q        object
A        object
label     int64
dtype: object


In [10]:
# ==========================================================
# 8. 질문 / 답변 분리
# ==========================================================


questions = data["Q"].tolist()

answers = data["A"].tolist()



print("질문 개수 :", len(questions))

print("답변 개수 :", len(answers))



print("\n샘플 데이터 확인")

print("Q :", questions[0])

print("A :", answers[0])

질문 개수 : 11823
답변 개수 : 11823

샘플 데이터 확인
Q : 12시 땡!
A : 하루가 또 가네요.


In [11]:
# ==========================================================
# 9. 문장 전처리 함수
# ==========================================================


def preprocess_sentence(sentence):


    # ------------------------------------------------------
    # 1) 영어 대문자를 소문자로 변경
    # ------------------------------------------------------

    sentence = sentence.lower()



    # ------------------------------------------------------
    # 2) 앞뒤 공백 제거
    # ------------------------------------------------------

    sentence = sentence.strip()



    # ------------------------------------------------------
    # 3) 문장부호 앞뒤에 공백 추가
    # 예:
    # 안녕? → 안녕 ?
    # ------------------------------------------------------

    sentence = re.sub(
        r"([?.!,])",
        r" \1 ",
        sentence
    )



    # ------------------------------------------------------
    # 4) 여러 개의 공백을 하나로 변경
    # ------------------------------------------------------

    sentence = re.sub(
        r'[" "]+',
        " ",
        sentence
    )



    # ------------------------------------------------------
    # 5) 한글, 영어, 기본 문장부호만 유지
    # ------------------------------------------------------

    sentence = re.sub(
        r"[^ㄱ-ㅎㅏ-ㅣ가-힣a-zA-Z?.!,]+",
        " ",
        sentence
    )



    # 마지막 공백 제거

    sentence = sentence.strip()



    return sentence

In [12]:
# ==========================================================
# 10. 전처리 적용
# ==========================================================


data["Q"] = data["Q"].apply(
    preprocess_sentence
)


data["A"] = data["A"].apply(
    preprocess_sentence
)



print("전처리 완료!")



data.head()

전처리 완료!


,Q,A,label
0,시 땡 !,하루가 또 가네요 .,0
1,지망 학교 떨어졌어,위로해 드립니다 .,0
2,박 일 놀러가고 싶다,여행은 언제나 좋죠 .,0
3,박 일 정도 놀러가고 싶다,여행은 언제나 좋죠 .,0
4,ppl 심하네,눈살이 찌푸려지죠 .,0


In [13]:
# ==========================================================
# 11. 전처리 결과 확인
# ==========================================================


print("Q :", data["Q"][0])

print("A :", data["A"][0])

Q : 시 땡 !
A : 하루가 또 가네요 .


In [14]:
# ==========================================================
# 12. Random Data Augmentation 함수
# ==========================================================


# ----------------------------------------------------------
# Random Deletion
#
# 문장 속 일부 단어를 랜덤하게 삭제
#
# 예)
# 오늘 너무 피곤해요
#
# ↓
#
# 오늘 피곤해요
# ----------------------------------------------------------


def random_deletion(
    sentence,
    delete_prob=0.1
):

    words = sentence.split()


    # 너무 짧은 문장은 변경하지 않음

    if len(words) <= 2:

        return sentence


    result = []


    for word in words:


        # delete_prob 확률로 단어 삭제

        if random.random() > delete_prob:

            result.append(word)



    # 모든 단어가 삭제되는 상황 방지

    if len(result) == 0:

        return random.choice(words)



    return " ".join(result)

In [15]:
# ==========================================================
# 13. Random Swap 함수
# ==========================================================


def random_swap(sentence):

    words = sentence.split()


    if len(words) <= 2:

        return sentence


    idx1, idx2 = random.sample(
        range(len(words)),
        2
    )


    words[idx1], words[idx2] = (
        words[idx2],
        words[idx1]
    )


    return " ".join(words)

In [16]:
# ==========================================================
# 14. 데이터 증강 실행
# ==========================================================


print(
    "원본 데이터 개수 :",
    len(data)
)



# 첫 번째 증강 데이터

aug_data_1 = data.copy()



aug_data_1["Q"] = aug_data_1["Q"].apply(
    lambda x: random_deletion(x)
)



# 두 번째 증강 데이터

aug_data_2 = data.copy()



aug_data_2["Q"] = aug_data_2["Q"].apply(
    lambda x: random_swap(x)
)



print("데이터 증강 완료")

원본 데이터 개수 : 11823
데이터 증강 완료


In [17]:
# ==========================================================
# 15. 데이터 병합
# ==========================================================


data_augmented = pd.concat(
    [
        data,
        aug_data_1,
        aug_data_2
    ]
)



# 완전히 같은 질문과 답변 제거

data_augmented = (
    data_augmented
    .drop_duplicates(
        subset=[
            "Q",
            "A"
        ]
    )
    .reset_index(drop=True)
)



print(
    "증강 후 데이터 개수 :",
    len(data_augmented)
)

증강 후 데이터 개수 : 24389


In [18]:
# ==========================================================
# 16. 증강 데이터 확인
# ==========================================================


for i in range(5):


    print(
        "원본 Q :",
        data.iloc[i]["Q"]
    )


    print(
        "증강 Q :",
        aug_data_1.iloc[i]["Q"]
    )


    print("-"*50)

원본 Q : 시 땡 !
증강 Q : 시 !
--------------------------------------------------
원본 Q : 지망 학교 떨어졌어
증강 Q : 지망 학교 떨어졌어
--------------------------------------------------
원본 Q : 박 일 놀러가고 싶다
증강 Q : 박 놀러가고
--------------------------------------------------
원본 Q : 박 일 정도 놀러가고 싶다
증강 Q : 박 일 놀러가고 싶다
--------------------------------------------------
원본 Q : ppl 심하네
증강 Q : ppl 심하네
--------------------------------------------------


In [19]:
# ==========================================================
# 17. SentencePiece 학습 데이터 생성
# ==========================================================


# SentencePiece 학습용 텍스트 파일 생성

SPM_INPUT_FILE = "chatbot_spm.txt"



with open(
    SPM_INPUT_FILE,
    "w",
    encoding="utf-8"
) as f:



    # 질문 데이터 저장

    for sentence in data_augmented["Q"]:

        f.write(
            sentence + "\n"
        )



    # 답변 데이터 저장

    for sentence in data_augmented["A"]:

        f.write(
            sentence + "\n"
        )



print(
    "SentencePiece 학습 데이터 생성 완료"
)

SentencePiece 학습 데이터 생성 완료


In [20]:
# ==========================================================
# 18. SentencePiece Tokenizer 학습 (수정)
# ==========================================================


VOCAB_SIZE = 8000



spm.SentencePieceTrainer.Train(

    f"--input={SPM_INPUT_FILE} "
    f"--model_prefix=chatbot_spm "
    f"--vocab_size={VOCAB_SIZE} "
    f"--pad_id=0 "
    f"--bos_id=1 "
    f"--eos_id=2 "
    f"--unk_id=3"

)



print(
    "SentencePiece 학습 완료"
)

SentencePiece 학습 완료


sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=chatbot_spm.txt --model_prefix=chatbot_spm --vocab_size=8000 --pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: chatbot_spm.txt
  input_format: 
  model_prefix: chatbot_spm
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk

In [21]:
# ==========================================================
# 19. SentencePiece 모델 로드
# ==========================================================


sp = spm.SentencePieceProcessor()



sp.Load(
    "chatbot_spm.model"
)



print(
    "Tokenizer Load 완료"
)

Tokenizer Load 완료


In [22]:
# ==========================================================
# 20. Token 변환 테스트
# ==========================================================


test_sentence = "오늘 날씨 어때?"



# 문장 → Token

tokens = sp.encode_as_pieces(
    test_sentence
)



# 문장 → 숫자 ID

token_ids = sp.encode_as_ids(
    test_sentence
)



print(
    "원본 문장 :",
    test_sentence
)



print(
    "Token :",
    tokens
)



print(
    "Token ID :",
    token_ids
)

원본 문장 : 오늘 날씨 어때?
Token : ['▁오늘', '▁날씨', '▁어때', '?']
Token ID : [85, 789, 320, 7997]


In [23]:
# ==========================================================
# 21. Special Token 확인
# ==========================================================


print(
    "PAD ID :",
    sp.pad_id()
)


print(
    "BOS ID :",
    sp.bos_id()
)


print(
    "EOS ID :",
    sp.eos_id()
)


print(
    "UNK ID :",
    sp.unk_id()
)

PAD ID : 0
BOS ID : 1
EOS ID : 2
UNK ID : 3


In [24]:
# ==========================================================
# 22. ChatbotDataset 클래스 구현
# ==========================================================


class ChatbotDataset(Dataset):


    def __init__(
        self,
        dataframe,
        tokenizer,
        max_len=40
    ):


        # 데이터 저장

        self.data = dataframe.reset_index(
            drop=True
        )


        # SentencePiece tokenizer

        self.tokenizer = tokenizer


        # 최대 길이

        self.max_len = max_len



    def __len__(self):


        # 데이터 개수 반환

        return len(self.data)



    def __getitem__(
        self,
        idx
    ):


        # --------------------------------------
        # 질문(Q), 답변(A) 가져오기
        # --------------------------------------

        question = self.data.loc[
            idx,
            "Q"
        ]


        answer = self.data.loc[
            idx,
            "A"
        ]



        # --------------------------------------
        # SentencePiece Token 변환
        # --------------------------------------

        src_ids = (

            [self.tokenizer.bos_id()]

            +

            self.tokenizer.encode_as_ids(
                question
            )

            +

            [self.tokenizer.eos_id()]

        )



        trg_ids = (

            [self.tokenizer.bos_id()]

            +

            self.tokenizer.encode_as_ids(
                answer
            )

            +

            [self.tokenizer.eos_id()]

        )



        # --------------------------------------
        # 길이 제한
        # --------------------------------------

        src_ids = src_ids[
            :self.max_len
        ]


        trg_ids = trg_ids[
            :self.max_len
        ]



        return (

            torch.tensor(
                src_ids,
                dtype=torch.long
            ),

            torch.tensor(
                trg_ids,
                dtype=torch.long
            )

        )

In [25]:
# ==========================================================
# 23. Dataset 생성 테스트
# ==========================================================


MAX_LEN = 40



dataset = ChatbotDataset(
    data_augmented,
    sp,
    MAX_LEN
)



print(
    "Dataset 크기 :",
    len(dataset)
)



src, trg = dataset[0]



print(
    "Encoder 입력 :",
    src
)


print(
    "Decoder 입력 :",
    trg
)

Dataset 크기 : 24389
Encoder 입력 : tensor([   1, 1168,    5, 7734,   53,    2])
Decoder 입력 : tensor([  1, 322,  11,  93, 123,  23,   4,   2])


In [26]:
# ==========================================================
# 24. Padding 처리 함수
# ==========================================================


def collate_fn(batch):


    # batch 안에는:
    #
    # (
    #  질문 tensor,
    #  답변 tensor
    # )
    #
    # 형태의 데이터가 들어있음


    src_batch = []

    trg_batch = []



    for src, trg in batch:


        src_batch.append(src)

        trg_batch.append(trg)



    # --------------------------------------
    # Padding 적용
    #
    # batch_first=True
    #
    # 결과 형태:
    #
    # (batch_size, sequence_length)
    # --------------------------------------


    src_batch = torch.nn.utils.rnn.pad_sequence(

        src_batch,

        batch_first=True,

        padding_value=sp.pad_id()

    )



    trg_batch = torch.nn.utils.rnn.pad_sequence(

        trg_batch,

        batch_first=True,

        padding_value=sp.pad_id()

    )



    return (

        src_batch,

        trg_batch

    )

In [27]:
# ==========================================================
# 25. DataLoader 생성
# ==========================================================


BATCH_SIZE = 64



dataloader = DataLoader(

    dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    collate_fn=collate_fn

)



print(
    "DataLoader 생성 완료"
)

DataLoader 생성 완료


In [28]:
# ==========================================================
# 26. Batch 데이터 확인
# ==========================================================


src_batch, trg_batch = next(
    iter(dataloader)
)



print(
    "Encoder 입력 크기:"
)

print(
    src_batch.shape
)



print(
    "\nDecoder 입력 크기:"
)

print(
    trg_batch.shape
)

Encoder 입력 크기:
torch.Size([64, 14])

Decoder 입력 크기:
torch.Size([64, 25])


In [29]:
# ==========================================================
# 27. Padding Token 확인
# ==========================================================


print(
    src_batch[0]
)


print(
    "PAD ID :",
    sp.pad_id()
)

tensor([   1, 1416,   13,   73,  215,   31,  257,   17,    2,    0,    0,    0,
           0,    0])
PAD ID : 0


In [30]:
# ==========================================================
# 28. Transformer Hyperparameter 설정
# ==========================================================


# SentencePiece Vocabulary 크기

VOCAB_SIZE = sp.get_piece_size()



# 문장 최대 길이

MAX_LEN = 40



# Batch 크기

BATCH_SIZE = 64



# Embedding 차원

D_MODEL = 256



# Encoder / Decoder Layer 개수

N_LAYERS = 2



# Multi Head 개수

N_HEADS = 8



# Feed Forward 차원

D_FF = 512



# Dropout 비율

DROPOUT = 0.1



print(
    "Vocabulary Size :",
    VOCAB_SIZE
)

Vocabulary Size : 8000


In [31]:
# ==========================================================
# 29. Positional Encoding
# ==========================================================


class PositionalEncoding(nn.Module):


    def __init__(
        self,
        d_model,
        max_len=5000
    ):

        super().__init__()



        # 위치 정보를 저장할 행렬 생성

        pe = torch.zeros(
            max_len,
            d_model
        )



        # 위치 값

        position = torch.arange(
            0,
            max_len,
            dtype=torch.float
        ).unsqueeze(1)



        # 주기 조절 값

        div_term = torch.exp(

            torch.arange(
                0,
                d_model,
                2
            ).float()

            *

            (-np.log(10000.0) / d_model)

        )



        # sin 위치 정보

        pe[:,0::2] = torch.sin(
            position * div_term
        )



        # cos 위치 정보

        pe[:,1::2] = torch.cos(
            position * div_term
        )



        # Batch 차원 추가

        pe = pe.unsqueeze(0)



        # 학습하지 않는 값으로 저장

        self.register_buffer(
            "pe",
            pe
        )



    def forward(
        self,
        x
    ):


        return (

            x

            +

            self.pe[
                :,
                :x.size(1)
            ]

        )

In [32]:
# ==========================================================
# 30. Multi-Head Attention
# ==========================================================


class MultiHeadAttention(nn.Module):


    def __init__(
        self,
        d_model,
        n_heads
    ):


        super().__init__()



        assert d_model % n_heads == 0



        self.d_model = d_model

        self.n_heads = n_heads


        self.head_dim = (
            d_model //
            n_heads
        )



        # Query, Key, Value Linear

        self.q_linear = nn.Linear(
            d_model,
            d_model
        )


        self.k_linear = nn.Linear(
            d_model,
            d_model
        )


        self.v_linear = nn.Linear(
            d_model,
            d_model
        )



        # 최종 출력 Linear

        self.fc_out = nn.Linear(
            d_model,
            d_model
        )



    def forward(
        self,
        query,
        key,
        value,
        mask=None
    ):


        batch_size = query.shape[0]



        Q = self.q_linear(query)

        K = self.k_linear(key)

        V = self.v_linear(value)



        # Head 분리

        Q = Q.view(
            batch_size,
            -1,
            self.n_heads,
            self.head_dim
        ).permute(
            0,2,1,3
        )


        K = K.view(
            batch_size,
            -1,
            self.n_heads,
            self.head_dim
        ).permute(
            0,2,1,3
        )


        V = V.view(
            batch_size,
            -1,
            self.n_heads,
            self.head_dim
        ).permute(
            0,2,1,3
        )



        # Attention Score 계산

        energy = torch.matmul(

            Q,

            K.permute(
                0,
                1,
                3,
                2
            )

        ) / np.sqrt(
            self.head_dim
        )



        # Mask 적용

        if mask is not None:

            energy = energy.masked_fill(
                mask == 0,
                -1e10
            )



        attention = torch.softmax(
            energy,
            dim=-1
        )



        out = torch.matmul(
            attention,
            V
        )



        # Head 합치기

        out = out.permute(
            0,
            2,
            1,
            3
        ).contiguous()



        out = out.view(
            batch_size,
            -1,
            self.d_model
        )



        return self.fc_out(out)

In [33]:
# ==========================================================
# 31-1. Feed Forward Network
# ==========================================================


class FeedForward(nn.Module):


    def __init__(
        self,
        d_model,
        d_ff,
        dropout=0.1
    ):


        super().__init__()



        self.linear1 = nn.Linear(
            d_model,
            d_ff
        )


        self.linear2 = nn.Linear(
            d_ff,
            d_model
        )


        self.dropout = nn.Dropout(
            dropout
        )


        self.relu = nn.ReLU()



    def forward(
        self,
        x
    ):


        return self.linear2(

            self.dropout(

                self.relu(

                    self.linear1(x)

                )

            )

        )

In [34]:
# ==========================================================
# 31-2. Encoder Layer
# ==========================================================


class EncoderLayer(nn.Module):


    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout
    ):


        super().__init__()



        self.self_attn = MultiHeadAttention(
            d_model,
            n_heads
        )



        self.ffn = FeedForward(
            d_model,
            d_ff,
            dropout
        )



        self.norm1 = nn.LayerNorm(
            d_model
        )


        self.norm2 = nn.LayerNorm(
            d_model
        )



        self.dropout = nn.Dropout(
            dropout
        )



    def forward(
        self,
        src,
        src_mask
    ):


        # Self Attention

        attn_out = self.self_attn(

            src,
            src,
            src,
            src_mask

        )



        # Residual + Normalization

        src = self.norm1(

            src

            +

            self.dropout(attn_out)

        )



        # Feed Forward

        ffn_out = self.ffn(
            src
        )



        src = self.norm2(

            src

            +

            self.dropout(ffn_out)

        )



        return src

In [35]:
# ==========================================================
# 32. Decoder Layer
# ==========================================================


class DecoderLayer(nn.Module):


    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout
    ):


        super().__init__()



        # Decoder Self Attention

        self.self_attn = MultiHeadAttention(
            d_model,
            n_heads
        )



        # Encoder와 연결

        self.cross_attn = MultiHeadAttention(
            d_model,
            n_heads
        )



        self.ffn = FeedForward(
            d_model,
            d_ff,
            dropout
        )



        self.norm1 = nn.LayerNorm(
            d_model
        )


        self.norm2 = nn.LayerNorm(
            d_model
        )


        self.norm3 = nn.LayerNorm(
            d_model
        )


        self.dropout = nn.Dropout(
            dropout
        )



    def forward(
        self,
        trg,
        enc_src,
        trg_mask,
        src_mask
    ):



        # 1. Masked Self Attention

        self_attn = self.self_attn(

            trg,
            trg,
            trg,
            trg_mask

        )



        trg = self.norm1(

            trg

            +

            self.dropout(self_attn)

        )



        # 2. Encoder-Decoder Attention

        cross_attn = self.cross_attn(

            trg,
            enc_src,
            enc_src,
            src_mask

        )



        trg = self.norm2(

            trg

            +

            self.dropout(cross_attn)

        )



        # 3. Feed Forward

        ffn_out = self.ffn(
            trg
        )



        trg = self.norm3(

            trg

            +

            self.dropout(ffn_out)

        )



        return trg

In [36]:
# ==========================================================
# 33. Transformer Chatbot 모델
# ==========================================================


class TransformerChatbot(nn.Module):


    def __init__(self):


        super().__init__()



        # Token Embedding

        self.embedding = nn.Embedding(

            VOCAB_SIZE,

            D_MODEL

        )



        # Position Encoding

        self.pos_encoding = PositionalEncoding(

            D_MODEL

        )



        self.dropout = nn.Dropout(
            DROPOUT
        )



        # Encoder

        self.encoder_layers = nn.ModuleList(

            [

                EncoderLayer(

                    D_MODEL,

                    N_HEADS,

                    D_FF,

                    DROPOUT

                )

                for _ in range(N_LAYERS)

            ]

        )



        # Decoder

        self.decoder_layers = nn.ModuleList(

            [

                DecoderLayer(

                    D_MODEL,

                    N_HEADS,

                    D_FF,

                    DROPOUT

                )

                for _ in range(N_LAYERS)

            ]

        )



        # 출력층

        self.fc_out = nn.Linear(

            D_MODEL,

            VOCAB_SIZE

        )



    def make_trg_mask(
        self,
        trg
    ):


        trg_len = trg.shape[1]


        mask = torch.tril(

            torch.ones(

                trg_len,

                trg_len,

                device=trg.device

            )

        ).bool()



        return mask.unsqueeze(0).unsqueeze(0)



    def forward(
        self,
        src,
        trg,
        src_mask=None
    ):



        trg_mask = self.make_trg_mask(
            trg
        )



        # Encoder 입력

        src = self.dropout(

            self.pos_encoding(

                self.embedding(src)

            )

        )



        # Decoder 입력

        trg = self.dropout(

            self.pos_encoding(

                self.embedding(trg)

            )

        )



        # Encoder 실행

        for layer in self.encoder_layers:


            src = layer(
                src,
                src_mask
            )



        # Decoder 실행

        for layer in self.decoder_layers:


            trg = layer(

                trg,

                src,

                trg_mask,

                src_mask

            )



        return self.fc_out(trg)

In [37]:
# ==========================================================
# 33. 모델 생성 테스트
# ==========================================================


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)



model = TransformerChatbot().to(device)



print(
    "Model 생성 완료"
)

Model 생성 완료


In [38]:
# ==========================================================
# 34. Loss 함수 / Optimizer 설정
# ==========================================================


# Padding Token

PAD_IDX = sp.pad_id()



# Cross Entropy Loss

criterion = nn.CrossEntropyLoss(

    ignore_index=PAD_IDX

)



# Optimizer

optimizer = optim.Adam(

    model.parameters(),

    lr=0.0001,

    betas=(0.9,0.98),

    eps=1e-9

)



print(
    "Loss / Optimizer 설정 완료"
)

Loss / Optimizer 설정 완료


In [39]:
# ==========================================================
# 35. Transformer Training Loop
# ==========================================================


def train_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    device
):


    model.train()


    total_loss = 0



    for src, trg in tqdm(
        dataloader
    ):


        # GPU 이동

        src = src.to(device)

        trg = trg.to(device)



        optimizer.zero_grad()



        # ----------------------------------
        # Decoder 입력과 정답 분리
        #
        # 입력:
        # <BOS> 나는 학교
        #
        # 정답:
        # 나는 학교 간다 <EOS>
        #
        # ----------------------------------


        output = model(

            src,

            trg[:,:-1]

        )



        # output:
        # [batch, seq_len, vocab]


        output_dim = output.shape[-1]



        output = output.contiguous().view(

            -1,

            output_dim

        )



        trg_y = trg[:,1:].contiguous().view(

            -1

        )



        # Loss 계산

        loss = criterion(

            output,

            trg_y

        )



        # Backpropagation

        loss.backward()



        # Gradient clipping

        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            1

        )



        optimizer.step()



        total_loss += loss.item()



    return total_loss / len(dataloader)

In [40]:
# ==========================================================
# 36. Model Training
# ==========================================================


EPOCHS = 5



for epoch in range(EPOCHS):


    loss = train_epoch(

        model,

        dataloader,

        optimizer,

        criterion,

        device

    )



    print(

        f"Epoch {epoch+1}/{EPOCHS}"

        f"  Loss : {loss:.4f}"

    )

100%|██████████| 382/382 [00:13<00:00, 29.31it/s]


Epoch 1/5  Loss : 5.7008


100%|██████████| 382/382 [00:12<00:00, 31.32it/s]


Epoch 2/5  Loss : 4.8277


100%|██████████| 382/382 [00:12<00:00, 30.28it/s]


Epoch 3/5  Loss : 4.4518


100%|██████████| 382/382 [00:12<00:00, 30.78it/s]


Epoch 4/5  Loss : 4.1654


100%|██████████| 382/382 [00:12<00:00, 30.73it/s]


Epoch 5/5  Loss : 3.9189


In [41]:
# ==========================================================
# 37. 모델 저장
# ==========================================================


MODEL_PATH = "chatbot_transformer.pt"



torch.save(

    model.state_dict(),

    MODEL_PATH

)



print(
    "모델 저장 완료:",
    MODEL_PATH
)

모델 저장 완료: chatbot_transformer.pt


In [42]:
# ==========================================================
# 38. 학습 모델 불러오기
# ==========================================================


MODEL_PATH = "chatbot_transformer.pt"



model.load_state_dict(

    torch.load(

        MODEL_PATH,

        map_location=device

    )

)



model.to(device)



model.eval()



print(
    "모델 로드 완료"
)

모델 로드 완료


In [43]:
# ==========================================================
# 39. Chatbot 응답 생성 함수
# ==========================================================


def chatbot_response(
    sentence
):


    model.eval()



    # ----------------------------------
    # 입력 문장 전처리
    # ----------------------------------

    sentence = preprocess_sentence(
        sentence
    )



    # ----------------------------------
    # Encoder 입력 생성
    # ----------------------------------

    src_ids = (

        [sp.bos_id()]

        +

        sp.encode_as_ids(
            sentence
        )

        +

        [sp.eos_id()]

    )



    src = torch.tensor(

        src_ids,

        dtype=torch.long

    ).unsqueeze(0).to(device)



    # ----------------------------------
    # Decoder 시작
    # ----------------------------------

    trg = torch.tensor(

        [

            [sp.bos_id()]

        ],

        dtype=torch.long

    ).to(device)



    with torch.no_grad():


        for _ in range(MAX_LEN):


            output = model(

                src,

                trg

            )



            # 마지막 단어 선택

            next_token = output[:,-1,:].argmax(

                dim=-1

            ).item()



            # EOS면 종료

            if next_token == sp.eos_id():

                break



            # Decoder 입력 추가

            trg = torch.cat(

                [

                    trg,

                    torch.tensor(

                        [[next_token]],

                        device=device

                    )

                ],

                dim=1

            )



    # ----------------------------------
    # Token → 문장 변환
    # ----------------------------------

    result = sp.decode_ids(

        trg.squeeze().tolist()[1:]

    )


    return result

In [44]:
# ==========================================================
# 40. Chatbot 테스트
# ==========================================================


test_questions = [

    "오늘 날씨 어때?",

    "너무 피곤해",

    "영화 보고 싶어",

    "기분이 안 좋아"

]



for q in test_questions:


    answer = chatbot_response(q)


    print(
        "Q:",
        q
    )


    print(
        "A:",
        answer
    )


    print("-"*40)

Q: 오늘 날씨 어때?
A: 대한 두려움을 해보세요 .
----------------------------------------
Q: 너무 피곤해
A: 믿고 해보세요 .
----------------------------------------
Q: 영화 보고 싶어
A: 그건 좀 더 좋은 사람 만나보세요 .
----------------------------------------
Q: 기분이 안 좋아
A: 다른 사람마다를 해보세요 .
----------------------------------------


In [45]:
# ==========================================================
# 41. BLEU Score 평가
# ==========================================================


def calculate_bleu(
    dataframe,
    sample_num=100
):


    smoothie = SmoothingFunction().method4



    scores = []



    sample_data = dataframe.sample(

        n=min(

            sample_num,

            len(dataframe)

        ),

        random_state=42

    )



    for _, row in tqdm(

        sample_data.iterrows(),

        total=len(sample_data)

    ):



        question = row["Q"]

        reference = row["A"]



        prediction = chatbot_response(

            question

        )



        reference_tokens = reference.split()

        prediction_tokens = prediction.split()



        score = sentence_bleu(

            [reference_tokens],

            prediction_tokens,

            smoothing_function=smoothie

        )



        scores.append(score)



    return np.mean(scores)

In [46]:
# ==========================================================
# BLEU Score 출력
# ==========================================================


bleu = calculate_bleu(

    data_augmented,

    sample_num=50

)



print(

    f"평균 BLEU Score : {bleu:.4f}"

)

100%|██████████| 50/50 [00:02<00:00, 23.25it/s]

평균 BLEU Score : 0.0432


자연어 처리 모델은 단순히 모델 구조만 구현하는 것이 아니라 데이터 전처리, Tokenization, Dataset 구성, 학습 방식, 평가 방법이 모두 연결되어야 한다는 것을 배웠다.  

특히 Transformer 구조의 Attention이 문장 내 중요한 단어 관계를 학습하는 방식과 Encoder-Decoder 구조가 입력 문장을 기반으로 새로운 문장을 생성하는 과정을 직접 구현하면서 이해할 수 있었다.  

가장 어려웠던 부분은 데이터 처리와 Transformer 입력 형태를 맞추는 과정이었다. 특히 SentencePiece Tokenizer 결과와 Padding 처리를 Transformer 구조에 연결하는 과정에서 데이터 형태를 지속적으로 확인해야 했다.  

또한 Word2Vec 모델의 저장 형식 차이로 인해 모델 로드 과정에서 오류가 발생하였고, Word2Vec 전체 모델과 KeyedVectors 형식의 차이를 확인하여 적절한 로딩 방식을 적용하였다.  

Transformer 학습 과정에서는 Decoder 입력과 정답 데이터를 한 칸씩 이동시키는 Target Shift 개념이 중요하다는 것을 이해할 수 있었다.